<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/bit-flipper-rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN - Bit Flipper

In this notebook, we'll train a Recurrent Neural Network (RNN) to flip bits in sequences of arbitrary length. The model will learn to transform `0` to `1` and vice versa for any input sequence.


## Setup

In [ ]:
!pip install wandb tsilva-notebook-utils==0.0.13 > /dev/null

Loading required API keys and authentication tokens from Colab secrets into environment variables:


In [ ]:
from tsilva_notebook_utils.colab import load_secrets_into_env
load_secrets_into_env([
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

Configure the training parameters and model architecture:


In [ ]:
import os
from tsilva_notebook_utils.colab import notebook_id_from_title

def setup_config():
    # @markdown ### 🌱 Reproducibility Settings

    # @markdown Random seed for reproducibility
    seed = 42  # @param {type:"integer"}

    # @markdown ### 🧩 Dataset Settings

    # @markdown Total size of the synthetic dataset
    dataset_size = 10000  # @param {type:"integer"}

    # @markdown Minimum length of input sequence (time steps)
    min_seq_length = 2  # @param {type:"integer"}

    # @markdown Maximum length of input sequence (time steps)
    max_seq_length = 100  # @param {type:"integer"}

    # @markdown ### 🏋️ Training Settings

    # @markdown Number of training epochs
    n_epochs = 5  # @param {type:"integer"}

    # @markdown Batch size for training
    batch_size = 256  # @param {type:"integer"}

    # @markdown Learning rate for the optimizer
    learning_rate = 0.001  # @param {type:"number"}

    # @markdown Max gradient norm for gradient clipping (0 to disable)
    max_grad_norm = 0  # @param {type:"number"}

    # @markdown ### 🧠 Model Architecture Settings

    # @markdown Input size (number of input features)
    input_size = 1  # @param {type:"integer"}

    # @markdown Hidden layer size
    hidden_size = 16  # @param {type:"integer"}

    # @markdown Output size (number of output features)
    output_size = 1  # @param {type:"integer"}

    # @markdown Nonlinearity type for RNN
    nonlinearity = "tanh"  # @param ["tanh", "relu"]

    # Generate notebook id from notebook title
    os.environ["NOTEBOOK_ID"] = notebook_id_from_title()

    return {
        'seed': seed,
        'n_epochs': n_epochs,
        'batch_size': batch_size,
        'dataset_size': dataset_size,
        'min_seq_length': min_seq_length,
        'max_seq_length': max_seq_length,
        'learning_rate': learning_rate,
        'max_grad_norm': max_grad_norm,
        'input_size': input_size,
        'hidden_size': hidden_size,
        'output_size': output_size,
        'nonlinearity': nonlinearity
    }

CONFIG = setup_config()

First, let's create a synthetic dataset of binary sequences and their bit-flipped targets:


In [ ]:
import random
import torch
from torch.utils.data import Dataset, DataLoader

class BitFlipDataset(Dataset):
    def __init__(
        self,
        dataset_size=None,
        min_seq_length=None,
        max_seq_length=None
    ):
        # Use default values from CONFIG if arguments are not provided
        if dataset_size is None: dataset_size = CONFIG['dataset_size']
        if min_seq_length is None: min_seq_length = CONFIG['min_seq_length']
        if max_seq_length is None: max_seq_length = CONFIG['max_seq_length']

        self.data = []  # Initialize an empty list to store data samples

        # Generate the dataset
        for _ in range(dataset_size):
            # Randomly select a sequence length within the given range
            seq_len = random.randint(min_seq_length, max_seq_length)

            # Generate a random binary sequence (0s and 1s), shape: (seq_len, 1)
            X = torch.randint(0, 2, (seq_len, 1)).float()

            # Create the target sequence by flipping the bits (1 -> 0, 0 -> 1)
            Y = 1.0 - X

            # Append the input-output pair to the dataset
            self.data.append((X, Y))

    # Return the total number of samples in the dataset
    def __len__(self):
        return len(self.data)

    # Retrieve a sample by index
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = BitFlipDataset()
print(len(train_dataset)) # Dataset size
print("".join([str(x) for x in train_dataset[0][0].squeeze().int().tolist()])) # Sample input
print("".join([str(x) for x in train_dataset[0][1].squeeze().int().tolist()])) # Sample output

Now let's build a data loader for batch processing. Since sequences have different lengths, we need a custom collate function to pad sequences within each batch to the same length:


In [ ]:
import torch.nn as nn

def collate_fn(batch):
    seqs_x, seqs_y = zip(*batch)
    padded_x = nn.utils.rnn.pad_sequence(seqs_x, batch_first=True, padding_value=-1)
    padded_y = nn.utils.rnn.pad_sequence(seqs_y, batch_first=True, padding_value=-1)
    return padded_x, padded_y

collate_fn([
    (torch.tensor([2, 2, 2, 2], dtype=torch.long), torch.tensor([3, 3, 3, 3], dtype=torch.long)),
    (torch.tensor([1, 1], dtype=torch.long), torch.tensor([2, 2], dtype=torch.long)),
    (torch.tensor([3, 3, 3], dtype=torch.long), torch.tensor([4, 4, 4], dtype=torch.long))
])

Let's verify the data loader with our custom collate function works correctly:


In [ ]:
batch_size = 10
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
x, y = next(iter(train_loader))
print("Batch shape: " + str(x.shape))
for batch in train_loader:
    x, y = batch
    if -1 not in x: continue
    print("Sample padded X: " + str(x[0].squeeze()))
    break

Now we'll create our model. It consists of two key components:

1. An RNN module that processes the sequence and outputs hidden states
2. A linear layer that transforms these hidden states into our desired output

Let's demonstrate a forward pass through these modules:


In [ ]:
import torch
import torch.nn as nn

# Display configuration
for key in ['input_size', 'hidden_size', 'output_size', 'nonlinearity']:
    print(f"{key.replace('_', ' ').title()}: {CONFIG[key]}")

# Prepare input tensor
x = torch.tensor([[[1], [2]]]).float()
print(f"\nInput shape: {x.shape}")

# Initialize RNN with specified nonlinearity
rnn = nn.RNN(CONFIG['input_size'], CONFIG['hidden_size'],
             batch_first=True, nonlinearity=CONFIG['nonlinearity'])

# Forward pass
hidden_states, last_hidden_state = rnn(x)

# Display output shapes and values
print(f"\nHidden states shape: {hidden_states.shape}")
print(f"Last hidden state shape: {last_hidden_state.shape}")
print("Hidden states:\n", hidden_states)
print("Last hidden state:\n", last_hidden_state)

# Apply Linear layer to the last hidden state
linear = nn.Linear(CONFIG['hidden_size'], CONFIG['output_size'])
output = linear(last_hidden_state)

# Display final output
print(f"\nOutput shape: {output.shape}")
print("Output:\n", output)

Now let's implement the complete `BitFlipperRNN` model by integrating the RNN and linear components. This model will:
1. Process each input sequence through an RNN layer, producing hidden states for each time step
2. Transform these hidden states through a fully connected (linear) layer to generate predictions
3. Return both the predictions and hidden states, allowing us to inspect how information flows through the network:


In [ ]:
class BitFlipperRNN(nn.Module):
    def __init__(
        self,
        input_size=None,
        hidden_size=None,
        output_size=None,
        nonlinearity=None
    ):
        super().__init__()

        # Use default values from CONFIG if not provided
        if input_size is None: input_size = CONFIG['input_size']
        if hidden_size is None: hidden_size = CONFIG['hidden_size']
        if output_size is None: output_size = CONFIG['output_size']
        if nonlinearity is None: nonlinearity = CONFIG['nonlinearity']

        # Define an RNN layer with specified nonlinearity
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True, nonlinearity=nonlinearity)

        # Fully connected layer to map hidden state output to desired output size
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Pass the input through the RNN layer
        hidden_states, _ = self.rnn(x)  # out: (batch, seq_len, hidden_size)

        # Pass the RNN output through the fully connected layer
        output = self.fc(hidden_states)  # output: (batch, seq_len, output_size)

        return output, hidden_states

model = BitFlipperRNN()

Before training, let's initialize Weights & Biases (wandb) for experiment tracking:


In [ ]:
from tsilva_notebook_utils.wandb import init_with_defaults
init_with_defaults(CONFIG)

Let's train our bit-flipping model and track performance metrics:


In [ ]:
import torch
import torch.optim as optim
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm
import wandb

# Set model in training mode
model.train()

# Optionally: Watch the model to log gradients and model topology
wandb.watch(model, log="all")

# Define the loss function as Mean Squared Error loss
loss_fn = nn.MSELoss()

# Set learning rate from configuration
learning_rate = CONFIG['learning_rate']

# Initialize the Adam optimizer with model parameters and the learning rate
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Get the number of training epochs from configuration
n_epochs = CONFIG['n_epochs']

# Get max gradient norm for clipping (if enabled)
max_grad_norm = CONFIG['max_grad_norm']

# Training loop with progress bar using tqdm
with tqdm(range(n_epochs), desc="Training") as pbar:
    for epoch in pbar:
        losses = []
        accuracies = []

        # Iterate over batches of data from the training data loader
        for x_batch, y_batch in train_loader:
            # Forward pass: compute model predictions and capture hidden states
            outputs, hidden_states = model(x_batch)

            # Truncate outputs and y_batch to exclude padding values
            # (otherwise they would contribute the loss; we just wanted
            # those values there for matrix math to work in the model forward pass)
            mask = x_batch != -1
            masked_outputs = torch.masked_select(outputs, mask)
            masked_y_batch = torch.masked_select(y_batch, mask)

            # Calculate the loss
            loss = loss_fn(masked_outputs, masked_y_batch)
            losses.append(loss.item())

            # Backpropagation step
            optimizer.zero_grad()  # Clear previous gradients
            loss.backward()        # Compute gradients

            # Apply gradient clipping if enabled (max_grad_norm > 0)
            if max_grad_norm > 0:
                clip_grad_norm_(model.parameters(), max_grad_norm)

            optimizer.step()       # Update model parameters

            # Calculate the accuracy
            with torch.no_grad():
                accuracy = torch.numel(masked_outputs == masked_y_batch) / torch.numel(masked_outputs)
                accuracies.append(accuracy)

        # Compute average stats for this epoch
        avg_loss = sum(losses) / len(train_loader)
        avg_accuracy = sum(accuracies) / len(accuracies)

        # Log average loss to wandb
        wandb.log({'epoch': epoch + 1, 'loss': avg_loss, 'accuracy' : avg_accuracy})

        # Update the progress bar with the current epoch and average loss
        pbar.set_postfix({'epoch': epoch + 1, 'loss': f'{avg_loss:.6f}', 'accuracy': f'{avg_accuracy * 100:.2f}%'})

        if avg_accuracy == 1.0:
            print("\nTraining finished, model memorized dataset.")
            break

wandb.finish()


Test the trained model with custom input sequences:

* Enter binary sequences (e.g., `01101`) when prompted
* Type `exit` to end testing


In [ ]:
def manual_test():
    while True:
        sequence_s = input()
        if sequence_s == "exit": return
        x = torch.tensor(list(map(int, sequence_s))).unsqueeze(-1).float()
        with torch.no_grad(): outputs, _ = model(x)
        prediction = torch.round(torch.abs(outputs))
        prediction = "".join([str(x) for x in prediction.int().squeeze().tolist()])
        print(prediction)

# Uncomment this line to manually test predictions
# (commented by default to allow running `notify_and_disconnect_after_timeout` in next cell)
#manual_test()

Set up automatic notification and disconnect after idle timeout (to prevent unnecessary resource usage):


In [ ]:
from tsilva_notebook_utils.colab import notify_and_disconnect_after_timeout
notify_and_disconnect_after_timeout()